## MPS Magnetization — 2D ITF Trotter Evolution

Time-evolve a product state under the 2D transverse-field Ising model
using pepsy's `MpsOptimizer` with different modes and backends.

**Parameters** (matching `compare_exact_vs_color.py`):
- 4×4 square lattice, PBC
- H = J ΣZZ + h ΣX,  J = −1, h = 2
- 2nd-order Trotter with dt = 0.25
- Initial state: intermediate-temperature quench (arXiv:2503.20870)

In [ ]:
import math
import numpy as np
import pepsy as py
from pepsy import tensors as core
from helper import (
    build_lattice, build_initial_state, build_trotter_gates,
    build_mpo_z_sq, measure_energy, measure_z, measure_z_sq,
)

try:
    import torch
except Exception:
    torch = None

### Backend selection

Uncomment the backend you want to use.

In [ ]:
# ── Backend (pick one) ───────────────────────────────────────────────────────
backend = "cupy"
# backend = "torch"

if backend == "cupy":
    to_backend = core.backend_cupy(dtype="complex128")
elif backend == "torch":
    DEVICE = "cuda" if torch is not None and torch.cuda.is_available() else "cpu"
    to_backend = core.backend_torch(device=DEVICE, dtype=torch.complex128)
else:
    raise ValueError(f"Unknown backend: {backend}")

optimizer = core.build_optimizer(progbar=False, directory="cash/", parallel=True)

### Lattice & Hamiltonian

In [ ]:
# ── Physical parameters ──────────────────────────────────────────────────────
Lx, Ly = 4, 4
L = Lx * Ly
coupling_j = -1.0   # J
field_h = +2.0      # h
cyclic = True
lattice = "square"
dt = 0.25           # Trotter step size

# ── Build lattice ────────────────────────────────────────────────────────────
# mode: "snake" | "snake-row-major" | "row-major" | "col-major" |
#       "hilbert" | "hilbert-row-major" | "diag"
lat = build_lattice(Lx, Ly, coupling_j, field_h, cyclic=cyclic, lattice=lattice, mode="hilbert")
mpo_H = lat["mpo_H"]
edges_1d = lat["edges_1d"]
sites = lat["sites"]
mapper = lat["mapper"]

mapper.show(title="Hilbert-curve 2D → 1D mapping")
print(f"Lattice: {Lx}×{Ly}, L={L}, {'PBC' if cyclic else 'OBC'}")
print(f"H = J·ΣZZ + h·ΣX,  J={coupling_j}, h={field_h}")
print(f"Trotter dt = {dt}")

### Initial state

Product state |Ψ(θ)⟩ from arXiv:2503.20870:
θ = arcsin(h/(zJ)) + 2π/9

In [ ]:
# ── Initial state ────────────────────────────────────────────────────────────
theta_offset = 2 * math.pi / 9   # offset from arcsin(h/(Jz)); change to set custom angle

psi0, theta_paper = build_initial_state(L, coupling_j, field_h, theta_offset=theta_offset)
print(
    f"Initial state: |Ψ(θ)⟩ product state, "
    f"θ={theta_paper:.4f} rad ({math.degrees(theta_paper):.2f}°), "
    f"offset={theta_offset:.4f}"
)

### 2nd-order Trotter gates

Strang splitting: RX(h·dt/2) → RZZ(2J·dt) → RX(h·dt/2)

In [ ]:
# ── Build 2nd-order Trotter gate stream ──────────────────────────────────────
gates_trotter = build_trotter_gates(sites, edges_1d, field_h, coupling_j, dt, to_backend)
print(f"Trotter gates: {len(gates_trotter)} total")

### MPS evolution parameters

In [ ]:
# ── Evolution parameters ─────────────────────────────────────────────────────
n_steps = 20                    # number of Trotter steps
chi = 120                       # MPS bond dimension

mode = "dmrg"  # MPS optimization mode: "mpo", "dmrg", "svd", "swap", "exact"

# DMRG-specific (only used when mode="dmrg"):
n_iter = 5                      # inner FIT iterations per 2q gate
k_2q_batch = 1                  # batch this many 2q gates into one FIT window

print(f"n_steps = {n_steps}, T = {n_steps * dt}, chi = {chi}")
print(f"Mode: {mode}")

### Build Z² MPO

In [ ]:
# ── Build Z² MPO ─────────────────────────────────────────────────────────────
mpo_z_sq_offdiag, diagonal_shift, mpo_z = build_mpo_z_sq(Lx, Ly, mapper)
print(f"M = (1/L)ΣZ_i  →  MPO bond dim: {mpo_z.max_bond()}")
print(f"M² off-diag MPO bond dim: {mpo_z_sq_offdiag.max_bond()}")

### Run time evolution

In [ ]:
from tqdm import tqdm

mpo_H.apply_to_arrays(to_backend)
mpo_z.apply_to_arrays(to_backend)
mpo_z_sq_offdiag.apply_to_arrays(to_backend)

psi = psi0.copy()
psi.apply_to_arrays(to_backend)

engine = py.MpsOptimizer(psi, chi, mode=mode, contraction_opt=optimizer)
engine.set_gates(gates_trotter)

times = [0.0]
energy = [measure_energy(engine.p.copy(), mpo_H, L, optimizer)]
z_sq = [measure_z_sq(engine.p.copy(), mpo_z_sq_offdiag, diagonal_shift, optimizer)]
z_mag = [measure_z(engine.p.copy(), mpo_z, optimizer)]
fidelity_per_step = []

pbar = tqdm(range(n_steps), desc=mode)
for step in pbar:
    engine.run(progbar=False, mode=mode, fidelity_samples=10,
               n_iter=n_iter, k_2q_batch=k_2q_batch)

    t_now = (step + 1) * dt
    psi_t = engine.p.copy()
    times.append(t_now)
    energy.append(measure_energy(psi_t, mpo_H, L, optimizer))
    z_sq.append(measure_z_sq(psi_t, mpo_z_sq_offdiag, diagonal_shift, optimizer))
    z_mag.append(measure_z(psi_t, mpo_z, optimizer))

    loss = engine.losses[-1]
    fidelity_per_step.append(loss)
    pbar.set_postfix({"~F": f"{loss:.6f}", "E/L": f"{energy[-1]:.4f}",
                      "Z²": f"{z_sq[-1]:.4f}", "Z": f"{z_mag[-1]:.4f}"})

times = np.array(times)
energy = np.array(energy)
z_sq = np.array(z_sq)
z_mag = np.array(z_mag)

fidelity_per_step = np.array(fidelity_per_step)
print(f"\nDone. {len(times)} measurement points.")


### Plot results

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "lines.linewidth": 1.8,
    "lines.markersize": 4,
})

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharex=True)

axes[0].plot(times, energy, "o-", color="#1f77b4", markersize=3)
axes[0].set_ylabel(r"$E / L$")
axes[0].set_title("Energy per site")

axes[1].plot(times, z_sq, "s-", color="#d62728", markersize=3)
axes[1].set_ylabel(r"$\langle (\Sigma Z / L)^2 \rangle$")
axes[1].set_title("Magnetization squared")

axes[2].plot(times, z_mag, "^-", color="#2ca02c", markersize=3)
axes[2].set_ylabel(r"$\langle \Sigma Z / L \rangle$")
axes[2].set_title("Magnetization")

for ax in axes:
    ax.set_xlabel(r"$t$")
    ax.grid(True, alpha=0.25, linestyle="--")
    ax.xaxis.set_major_locator(MaxNLocator(6))
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle(
    f"MPS Trotter ({mode}) — {Lx}×{Ly} ITF, $dt$={dt}, $\\chi$={chi}, "
    f"{'PBC' if cyclic else 'OBC'}",
    fontsize=13, fontweight="bold",
)
plt.tight_layout()
plt.savefig("magnetization.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# ── Fidelity per Trotter step ────────────────────────────────────────────────
step_indices = np.arange(1, n_steps + 1)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(step_indices, fidelity_per_step, ">-", lw=1.2, markersize=4,
        color="#7b2cbf", alpha=0.85)
ax.axhline(fidelity_per_step.mean(), ls="--", lw=0.8, color="gray", alpha=0.6,
           label=f"mean = {fidelity_per_step.mean():.6f}")

# Auto y-limits to show variations clearly
ymin = fidelity_per_step.min() - 0.005
ymax = min(fidelity_per_step.max() + 0.005, 1.0)
ax.set_ylim(ymin, ymax)

ax.set_xlabel("Trotter step")
ax.set_ylabel("Fidelity")
ax.set_title(f"Fidelity per step — {mode}, $\\chi$={chi}, $dt$={dt}",
             fontweight="bold")
ax.grid(True, alpha=0.25, linestyle="--")
ax.set_xlim(0.5, n_steps + 0.5)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(loc="lower left", framealpha=0.9)
plt.tight_layout()
plt.show()

print(f"Fidelity per step: min={fidelity_per_step.min():.8f}, "
      f"max={fidelity_per_step.max():.8f}, mean={fidelity_per_step.mean():.8f}")

### Save results

In [ ]:
# import quimb as qu

# results = {
#     "times": times,
#     "energy": energy,
#     "z_sq": z_sq,
#     "z_mag": z_mag,
#     "losses": np.array(engine.losses),
#     "Lx": Lx, "Ly": Ly, "L": L,
#     "J": coupling_j, "h": field_h,
#     "dt": dt, "chi": chi, "mode": mode,
#     "cyclic": cyclic, "n_steps": n_steps,
# }

# save_path = f"store/mps_mag_L{L}_dt{dt}_c{cyclic}_chi{chi}_{mode}.pkl"
# qu.save_to_disk(results, save_path)
# print(f"Saved to {save_path}")

### Sample from final MPS

In [ ]:
from pepsy.sampling import MpsSampler

# Sample from the final MPS
sampler = MpsSampler(
    engine.p,
    one_d_to_two_d=lat["res"]["one_d_to_two_d"],
)
result = sampler.sample(n_samples=8, seed=2)

# Plot samples as 2D heatmaps
n_show = min(len(result), 8)
ncols = 8
nrows = (n_show + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 3.2 * nrows))
axes = np.atleast_2d(axes)

cmap = plt.cm.colors.ListedColormap(["#2196F3", "#FF5722"])  # blue=↑, red=↓

for idx in range(nrows * ncols):
    ax = axes[idx // ncols, idx % ncols]
    if idx < n_show:
        grid = result.configs_2d[idx]
        prob = result.probs[idx]
        ax.imshow(grid, cmap=cmap, vmin=0, vmax=1, aspect="equal")
        for iy in range(Ly):
            for ix in range(Lx):
                label = "↑" if grid[iy, ix] == 0 else "↓"
                ax.text(ix, iy, label, ha="center", va="center",
                        fontsize=14, fontweight="bold", color="white")
        ax.set_title(f"p = {prob:.3e}", fontsize=10)
        ax.set_xticks(range(Lx))
        ax.set_yticks(range(Ly))
        ax.set_xticklabels(range(Lx), fontsize=8)
        ax.set_yticklabels(range(Ly), fontsize=8)
        ax.set_xlabel("x", fontsize=9)
        ax.set_ylabel("y", fontsize=9)
    else:
        ax.axis("off")

fig.suptitle(
    f"Sampled spin configurations — {Lx}×{Ly} ITF, t={times[-1]:.2f}, χ={chi}",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.show()

# Summary
probs = np.array(result.probs)
mags = result.magnetizations()
print(f"Sampled {len(result)} configs | prob range: [{probs.min():.3e}, {probs.max():.3e}]")
print(f"Sample magnetizations: {mags}")